# Assignment 10: Score-Based Models — Advanced Topics

This notebook covers advanced topics in score-based generative modelling from Lecture 10, including:
- Multi-scale noise perturbation: training with a sequence of noise levels $\sigma_1 > \sigma_2 > \cdots > \sigma_L$
- Noise-conditional score networks: a single network $s_\theta(x, \sigma)$ for all noise levels
- Weighted score matching: down-weighting contributions from high-noise levels with $\lambda(\sigma)$
- Annealed Langevin dynamics: progressive refinement from coarse to fine
- Unifying score-based models and diffusion models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: Multi-Scale Noise Perturbation

A single noise level $\sigma$ is insufficient for learning a score function over all of $\mathbb{R}^D$:
- **Large $\sigma$** → score is reliable everywhere (low-density regions are filled in) but the perturbed distribution is blurry and far from the data
- **Small $\sigma$** → score is accurate near the data manifold but unreliable in gaps between modes

**Solution**: use a **geometric sequence** of $L$ noise levels $\sigma_1 > \sigma_2 > \cdots > \sigma_L$ and train a single network on all of them simultaneously.
The geometric schedule ensures exponentially finer resolution as $\sigma$ decreases:
$$\sigma_i = \sigma_{\max} \left(\frac{\sigma_{\min}}{\sigma_{\max}}\right)^{\frac{i-1}{L-1}}, \quad i = 1, \ldots, L$$

### 1.1 Geometric Noise Schedule

In [ ]:
def get_noise_schedule(sigma_max=5.0, sigma_min=0.01, L=10):
    """Geometric sequence of noise levels from sigma_max down to sigma_min.

    sigma_i = sigma_max * (sigma_min / sigma_max)^(i/(L-1))  for i = 0, ..., L-1

    Args:
        sigma_max : float, largest noise level
        sigma_min : float, smallest noise level
        L         : int, number of noise levels

    Returns:
        sigmas : np.ndarray (L,), descending noise levels
    """
    # TODO:
    # return sigma_max * (sigma_min / sigma_max) ** (np.arange(L) / (L - 1))
    return sigma_max * (sigma_min / sigma_max) ** (np.arange(L) / (L - 1))

In [ ]:
# Sanity check: first element ≈ sigma_max, last ≈ sigma_min
sigmas = get_noise_schedule(5.0, 0.01, 10)
print(f'Noise schedule (L=10): {sigmas.round(4)}')
print(f'First element: {sigmas[0]:.4f}  (expected ≈ 5.0)')
print(f'Last  element: {sigmas[-1]:.4f}  (expected ≈ 0.01)')
print(f'Ratio between consecutive levels: {(sigmas[1:] / sigmas[:-1]).round(4)}')

### 1.2 Perturbing Data at All Noise Levels

In [ ]:
def perturb_data(X, sigmas):
    """Perturb data at all noise levels.

    Args:
        X      : np.ndarray (N, D)
        sigmas : np.ndarray (L,)

    Returns:
        Z : np.ndarray (L, N, D), perturbed data: Z[i] = X + sigmas[i] * eps
    """
    # TODO:
    # Z = np.zeros((len(sigmas), *X.shape))
    # for i, sigma in enumerate(sigmas):
    #     eps = np.random.randn(*X.shape)
    #     Z[i] = X + sigma * eps
    # return Z
    Z = np.zeros((len(sigmas), *X.shape))
    for i, sigma in enumerate(sigmas):
        eps = np.random.randn(*X.shape)
        Z[i] = X + sigma * eps
    return Z

In [ ]:
# Generate 2D bimodal data: mixture of N((-2,-2), 0.3²I) and N((2,2), 0.3²I)
np.random.seed(42)
N = 1000
half = N // 2
X_data = np.vstack([
    np.random.randn(half, 2) * 0.3 + np.array([-2.0, -2.0]),
    np.random.randn(half, 2) * 0.3 + np.array([ 2.0,  2.0]),
])

# Sanity check: shape and statistics
sigmas_demo = np.array([2.0, 1.0, 0.5, 0.1])
Z = perturb_data(X_data, sigmas_demo)
print(f'X_data shape: {X_data.shape}')
print(f'Z shape     : {Z.shape}  (expected (4, {N}, 2))')
for i, sig in enumerate(sigmas_demo):
    print(f'  sigma={sig:.2f}  ->  Z[{i}] std ≈ {Z[i].std():.3f}')

In [ ]:
# Visualise: perturbed distributions for sigma in {2.0, 1.0, 0.5, 0.1}
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].scatter(X_data[:, 0], X_data[:, 1], s=4, alpha=0.5, color='steelblue')
axes[0].set_title('Original data')
axes[0].set_xlim(-7, 7); axes[0].set_ylim(-7, 7)

for ax, sig, Zi in zip(axes[1:], sigmas_demo, Z):
    ax.scatter(Zi[:, 0], Zi[:, 1], s=4, alpha=0.4, color='coral')
    ax.set_title(f'$\\sigma={sig}$')
    ax.set_xlim(-7, 7); ax.set_ylim(-7, 7)

plt.suptitle('Multi-scale noise perturbation of 2D bimodal data')
plt.tight_layout()
plt.show()

---
## Part 2: Noise-Conditional Score Network (PyTorch)

Instead of training $L$ separate score networks (one per noise level), we train a **single** network
$$s_\theta(x, \sigma) \approx \nabla_x \log q_\sigma(x)$$
that takes the noise level $\sigma$ as an additional input. This approach:
- Enables **smooth interpolation** between noise levels
- Shares parameters across levels, improving data efficiency
- Allows sampling with any intermediate $\sigma$ not seen during training

We condition on $\log(\sigma)$ (rather than $\sigma$ directly) to give the network a more linear view of the geometric scale.

In [ ]:
import torch
import torch.nn as nn

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
class NoisyScoreNet(nn.Module):
    """Noise-conditional score network s_theta(x, sigma).

    Architecture:
        Input: concatenate x (D-dim) and log(sigma) (1-dim)
        -> Linear(D+1, H) -> SiLU -> Linear(H, H) -> SiLU -> Linear(H, D)

    The output is divided by sigma so that the network internally predicts
    sigma * s_theta(x, sigma), keeping inputs and outputs on similar scales
    regardless of noise level.
    """

    def __init__(self, data_dim=2, hidden_dim=128):
        super().__init__()
        # TODO:
        # self.net = nn.Sequential(
        #     nn.Linear(data_dim + 1, hidden_dim),
        #     nn.SiLU(),
        #     nn.Linear(hidden_dim, hidden_dim),
        #     nn.SiLU(),
        #     nn.Linear(hidden_dim, data_dim),
        # )
        raise NotImplementedError

    def forward(self, x, sigma):
        """
        Args:
            x     : Tensor (B, D)
            sigma : Tensor (B,) or float

        Returns:
            score : Tensor (B, D), predicted grad_x log q_sigma(x)
        """
        # TODO:
        # sigma_t = sigma * torch.ones(x.shape[0], device=x.device)
        # log_sigma = torch.log(sigma_t).view(-1, 1)
        # x_in = torch.cat([x, log_sigma], dim=1)
        # return self.net(x_in) / sigma_t.view(-1, 1)
        raise NotImplementedError

In [ ]:
# Sanity check: instantiate and check output shapes
# (will raise NotImplementedError until implemented)
model_test = NoisyScoreNet(data_dim=2, hidden_dim=128).to(device)
x_test = torch.randn(16, 2).to(device)
sigma_test = torch.full((16,), 1.0).to(device)
score_test = model_test(x_test, sigma_test)
print(f'Input  shape: {x_test.shape}')
print(f'Output shape: {score_test.shape}  (expected torch.Size([16, 2]))')
n_params = sum(p.numel() for p in model_test.parameters())
print(f'Number of parameters: {n_params}')

---
## Part 3: Weighted DSM Objective

The **weighted score matching** objective trains across all noise levels simultaneously:
$$\mathcal{L}_{\text{weighted}}(\theta) = \sum_{i=1}^{L} \lambda(\sigma_i) \cdot \mathbb{E}_{x \sim p,\, \varepsilon \sim \mathcal{N}(0,I)}\!\left[\|\sigma_i \cdot s_\theta(x + \sigma_i \varepsilon,\, \sigma_i) + \varepsilon\|^2\right]$$

The natural choice $\lambda(\sigma) = \sigma^2$ ensures each noise level contributes equally — it corresponds to unit Fisher information weighting.

**Key insight**: the optimal score satisfies
$$\nabla_z \log q_{\sigma}(z \mid x) = -\frac{z - x}{\sigma^2} = -\frac{\varepsilon}{\sigma}$$
so the residual $\sigma \cdot s_\theta + \varepsilon$ is zero at the optimum for every noise level.

In [ ]:
def weighted_dsm_loss(model, X_batch, sigmas, lambdas=None):
    """Weighted DSM loss across all noise levels.

    L = (1/L) * sum_i lambda_i * E_{eps} [|| sigma_i * s_theta(x + sigma_i*eps, sigma_i) + eps ||^2]

    Args:
        model   : NoisyScoreNet
        X_batch : Tensor (B, D), clean data
        sigmas  : Tensor (L,), noise levels (all on device)
        lambdas : Tensor (L,) or None (if None use sigma^2 weighting)

    Returns:
        loss      : scalar Tensor
        per_level : Tensor (L,), loss per noise level (for monitoring)
    """
    if lambdas is None:
        lambdas = sigmas ** 2

    # TODO:
    # per_level_losses = []
    # for i, (sigma_i, lambda_i) in enumerate(zip(sigmas, lambdas)):
    #     eps = torch.randn_like(X_batch)
    #     z = X_batch + sigma_i * eps                      # noisy sample
    #     sigma_in = sigma_i * torch.ones(len(X_batch), device=X_batch.device)
    #     score = model(z, sigma_in)                        # s_theta(z, sigma)
    #     residual = sigma_i * score + eps                  # should be 0 at optimum
    #     per_level_losses.append(lambda_i * (residual**2).sum(dim=1).mean())
    # loss = torch.stack(per_level_losses).mean()
    # return loss, torch.stack(per_level_losses).detach()
    raise NotImplementedError

In [ ]:
# Sanity check: loss should be a scalar, per_level should have shape (L,)
# (requires NoisyScoreNet and weighted_dsm_loss to be implemented)
sigmas_t = torch.tensor(get_noise_schedule(5.0, 0.01, 5), dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_data[:32], dtype=torch.float32).to(device)
loss_test, per_level_test = weighted_dsm_loss(model_test, X_test_t, sigmas_t)
print(f'Loss value  : {loss_test.item():.4f}')
print(f'Per-level   : {per_level_test.cpu().numpy().round(4)}')
print(f'Per-level shape: {per_level_test.shape}  (expected torch.Size([5]))')

---
## Part 4: Joint Training on All Noise Levels

We now train the noise-conditional score network jointly across all noise levels using the weighted DSM loss.
A key observation during training is that **high-$\sigma$ levels converge faster** — the score landscape is smoother and the target signal is stronger. Low-$\sigma$ levels are harder and converge later.

In [ ]:
def train_noisy_score_net(X_data, sigmas_np, n_epochs=400, lr=1e-3, batch_size=256):
    """Train NoisyScoreNet with weighted DSM loss across all noise levels.

    Args:
        X_data    : np.ndarray (N, 2)
        sigmas_np : np.ndarray (L,), noise levels
        n_epochs  : int
        lr        : float
        batch_size: int

    Returns:
        model             : trained NoisyScoreNet
        loss_history      : list of float, total loss per epoch
        per_level_history : np.ndarray (n_epochs, L), per-level loss history
    """
    X_t = torch.tensor(X_data, dtype=torch.float32).to(device)
    sigmas_t = torch.tensor(sigmas_np, dtype=torch.float32).to(device)
    model = NoisyScoreNet(data_dim=2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_t), batch_size=batch_size, shuffle=True)

    loss_history = []
    per_level_history = []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_per_level = torch.zeros(len(sigmas_np), device=device)

        for (xb,) in loader:
            # TODO:
            # 1. opt.zero_grad()
            # 2. loss, per_level = weighted_dsm_loss(model, xb, sigmas_t)
            # 3. loss.backward(); opt.step()
            # 4. epoch_loss += loss.item()
            # 5. epoch_per_level += per_level
            raise NotImplementedError

        loss_history.append(epoch_loss / len(loader))
        per_level_history.append((epoch_per_level / len(loader)).cpu().numpy())
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1:4d}  loss={loss_history[-1]:.4f}')

    return model, loss_history, np.array(per_level_history)

In [ ]:
# Train the model (requires NoisyScoreNet, weighted_dsm_loss, and train_noisy_score_net to be implemented)
np.random.seed(42)
torch.manual_seed(42)

sigmas_train = get_noise_schedule(sigma_max=5.0, sigma_min=0.05, L=10)
model, loss_history, per_level_history = train_noisy_score_net(
    X_data, sigmas_train, n_epochs=400, lr=1e-3, batch_size=256
)
print(f'\nFinal loss: {loss_history[-1]:.4f}')

In [ ]:
# Plot: per-level loss curves vs epoch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_history, color='black', lw=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Weighted DSM loss')
axes[0].set_title('Total training loss')
axes[0].set_yscale('log')

for i, sig in enumerate(sigmas_train):
    axes[1].plot(per_level_history[:, i], label=f'$\\sigma={sig:.3f}$', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Per-level loss')
axes[1].set_title('Loss per noise level (high-$\\sigma$ converges faster)')
axes[1].legend(fontsize=7, ncol=2)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Plot: learned score fields for sigma in {2.0, 0.5, 0.1} (quiver plots)
def plot_score_field(model, sigma, ax, grid_n=20, xlim=(-5, 5), ylim=(-5, 5)):
    """Plot the score field s_theta(x, sigma) as a quiver plot."""
    xs = np.linspace(*xlim, grid_n)
    ys = np.linspace(*ylim, grid_n)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.stack([XX.ravel(), YY.ravel()], axis=1)

    model.eval()
    with torch.no_grad():
        x_t = torch.tensor(grid, dtype=torch.float32).to(device)
        sig_t = torch.full((len(grid),), sigma, device=device)
        scores = model(x_t, sig_t).cpu().numpy()

    U = scores[:, 0].reshape(grid_n, grid_n)
    V = scores[:, 1].reshape(grid_n, grid_n)
    ax.quiver(XX, YY, U, V, alpha=0.7)
    ax.scatter(X_data[:, 0], X_data[:, 1], s=5, alpha=0.3, color='steelblue', zorder=2)
    ax.set_title(f'Score field  $\\sigma={sigma}$')
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)

sigmas_plot = [2.0, 0.5, 0.1]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, sig in zip(axes, sigmas_plot):
    plot_score_field(model, sig, ax)
plt.suptitle('Learned noise-conditional score fields')
plt.tight_layout()
plt.show()

---
## Part 5: Annealed Langevin Dynamics

**Annealed Langevin Dynamics (ALD)** uses the noise-conditional score network to progressively refine samples, moving from coarse structure (large $\sigma$) to fine detail (small $\sigma$).

For each noise level $\sigma_i$ (in descending order), repeat $T$ Langevin steps:
$$x \leftarrow x + \frac{\varepsilon_i}{2} \cdot s_\theta(x, \sigma_i) + \sqrt{\varepsilon_i}\, \mathcal{N}(0, I)$$

where the step size $\varepsilon_i \propto \sigma_i^2$ decreases with noise level. The annealing is crucial: starting with large $\sigma$ moves samples into high-density regions globally, while small $\sigma$ refines the fine structure.

In [ ]:
def annealed_langevin(model, x_init, sigmas, n_steps_per_level=100, alpha=0.1):
    """Run Annealed Langevin Dynamics across decreasing noise levels.

    For each sigma (in descending order):
        step_size = alpha * sigma^2 / sigma_L^2   (scaled to smallest level)
        for k in range(n_steps_per_level):
            z_noise = np.random.randn(*x.shape)
            x = x + (step_size/2) * score(x, sigma) + sqrt(step_size) * z_noise

    Args:
        model             : NoisyScoreNet (placed in eval mode internally)
        x_init            : np.ndarray (N, D), starting samples
        sigmas            : np.ndarray (L,), noise levels in DESCENDING order
        n_steps_per_level : int, Langevin steps per noise level
        alpha             : float, base step size scaling

    Returns:
        x_final   : np.ndarray (N, D)
        snapshots : list of np.ndarray (N, D), one snapshot per noise level
    """
    x = x_init.copy().astype(float)
    snapshots = []
    sigma_min = sigmas[-1]

    # TODO:
    # model.eval()
    # for sigma in sigmas:
    #     step_size = alpha * (sigma / sigma_min)**2 * sigma_min**2
    #     sigma_t = torch.tensor([sigma], dtype=torch.float32).to(device)
    #     for _ in range(n_steps_per_level):
    #         x_t = torch.tensor(x, dtype=torch.float32).to(device)
    #         with torch.no_grad():
    #             score = model(x_t, sigma_t.expand(len(x))).cpu().numpy()
    #         z = np.random.randn(*x.shape)
    #         x = x + (step_size / 2) * score + np.sqrt(step_size) * z
    #     snapshots.append(x.copy())
    # return x, snapshots
    model.eval()
    for sigma in sigmas:
        step_size = alpha * (sigma / sigma_min)**2 * sigma_min**2
        sigma_t = torch.tensor([sigma], dtype=torch.float32).to(device)
        for _ in range(n_steps_per_level):
            x_t = torch.tensor(x, dtype=torch.float32).to(device)
            with torch.no_grad():
                score = model(x_t, sigma_t.expand(len(x))).cpu().numpy()
            z = np.random.randn(*x.shape)
            x = x + (step_size / 2) * score + np.sqrt(step_size) * z
        snapshots.append(x.copy())
    return x, snapshots

In [ ]:
# Run ALD: start from N(0, sigma_max^2) and progressively refine
np.random.seed(0)
N_samples = 500
x_init = np.random.randn(N_samples, 2) * sigmas_train[0]

x_final, snapshots = annealed_langevin(
    model, x_init, sigmas_train, n_steps_per_level=100, alpha=0.1
)

print(f'x_init  mean={x_init.mean(axis=0).round(3)}, std={x_init.std(axis=0).round(3)}')
print(f'x_final mean={x_final.mean(axis=0).round(3)}, std={x_final.std(axis=0).round(3)}')

In [ ]:
# 4-panel plot: initial noise -> after sigma=2.0 -> after sigma=0.5 -> after sigma=0.1
# Find indices closest to target sigmas
target_sigmas = [2.0, 0.5, 0.1]
snap_indices = [np.argmin(np.abs(sigmas_train - s)) for s in target_sigmas]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

axes[0].scatter(x_init[:, 0], x_init[:, 1], s=8, alpha=0.5, color='gray')
axes[0].set_title(f'Initial noise\n($\\sigma_{{max}}={sigmas_train[0]:.1f}$)')

for ax, idx, ts in zip(axes[1:-1], snap_indices, target_sigmas):
    ax.scatter(snapshots[idx][:, 0], snapshots[idx][:, 1], s=8, alpha=0.5, color='coral')
    ax.set_title(f'After $\\sigma \\approx {ts}$')

axes[-1].scatter(x_final[:, 0], x_final[:, 1], s=8, alpha=0.5, color='steelblue', label='ALD samples')
axes[-1].scatter(X_data[:, 0], X_data[:, 1], s=8, alpha=0.2, color='orange', label='True data')
axes[-1].set_title('Final samples vs data')
axes[-1].legend(fontsize=7)

for ax in axes:
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)

plt.suptitle('Annealed Langevin Dynamics: progressive refinement')
plt.tight_layout()
plt.show()

---
## Part 6: Connection to Diffusion Models

Score-based models and diffusion models are **the same framework** viewed from different angles.

| Framework | Training objective |
|-----------|-------------------|
| **DDPM** (Assignment 6) | $\mathbb{E}\bigl[\|\varepsilon - \varepsilon_\theta(z_t, t)\|^2\bigr]$ |
| **Score-based** | $\mathbb{E}\bigl[\|s_\theta(z, \sigma) - \nabla_z \log q_\sigma(z \mid x)\|^2\bigr]$ |

These are **equivalent** because the optimal score satisfies:
$$\nabla_z \log q_\sigma(z \mid x) = -\frac{z - x}{\sigma^2} = -\frac{\varepsilon}{\sigma}$$

Therefore: $\varepsilon_\theta = -\sigma \cdot s_\theta$ — the two parameterisations differ only by a sign and a scale of $\sigma$.

In [ ]:
def dsm_to_ddpm_equivalence(n_samples=5000, sigma=0.5):
    """Numerically verify that DSM loss and DDPM noise-prediction loss are equivalent.

    For a 1D standard normal target p(x) = N(0,1):
    - DSM target at sigma: -(z - x) / sigma^2 = -eps / sigma
    - DDPM target: eps

    Show that minimising either gives the same optimal network output.

    Args:
        n_samples : int
        sigma     : float

    Returns:
        None (produces plots and prints)
    """
    np.random.seed(0)
    x_data = np.random.randn(n_samples)
    eps = np.random.randn(n_samples)
    z = x_data + sigma * eps

    # TODO:
    # 1. Compute the DSM target at z: -(z - x_data) / sigma^2 = -eps / sigma
    dsm_target = -(z - x_data) / sigma**2   # = -eps / sigma
    # 2. Compute the DDPM target: eps
    ddpm_target = eps
    # 3. Show that -sigma * dsm_target = ddpm_target (they differ only by -sigma)
    equivalent = np.allclose(-sigma * dsm_target, ddpm_target)
    print(f'sigma = {sigma}')
    print(f'DSM  target:  mean={dsm_target.mean():.4f},  std={dsm_target.std():.4f}')
    print(f'DDPM target:  mean={ddpm_target.mean():.4f}, std={ddpm_target.std():.4f}')
    print(f'-sigma * DSM_target == DDPM_target: {equivalent}')
    print(f'  (relationship: eps_theta = -sigma * s_theta)')

    # 4. Plot: histogram of DSM and DDPM targets at z ≈ 0
    mask = np.abs(z) < 0.2
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(dsm_target[mask], bins=40, density=True, alpha=0.6, color='steelblue', label='DSM target')
    x_plot = np.linspace(-10, 10, 300)
    axes[0].plot(x_plot, norm.pdf(x_plot, 0, 1 / sigma), 'r--', lw=2, label=f'N(0, 1/σ²)')
    axes[0].set_title(f'DSM targets at $z \\approx 0$ ($\\sigma={sigma}$)')
    axes[0].legend()

    axes[1].hist(ddpm_target[mask], bins=40, density=True, alpha=0.6, color='coral', label='DDPM target')
    axes[1].plot(x_plot, norm.pdf(x_plot, 0, 1), 'r--', lw=2, label='N(0,1)')
    axes[1].set_title(f'DDPM targets at $z \\approx 0$ ($\\sigma={sigma}$)')
    axes[1].legend()

    plt.suptitle('DSM vs DDPM: equivalent objectives up to sign and scale')
    plt.tight_layout()
    plt.show()


dsm_to_ddpm_equivalence(n_samples=5000, sigma=0.5)

**Question 1:** The weighting $\lambda(\sigma) = \sigma^2$ balances contributions from different noise levels. Explain why a uniform weight $\lambda(\sigma) = 1$ would be problematic.

**Your answer**: TODO

**Question 2:** How do score-based models (annealed Langevin) and diffusion models (DDPM) differ in their sampling procedure? What is the key practical advantage of diffusion models?

**Your answer**: TODO

---
## Part 7 (Bonus): Continuous Noise Levels

In the limit $L \to \infty$, the discrete noise schedule becomes a **continuous** function $\sigma: [0,1] \to \mathbb{R}_+$.
This is the perspective of Song et al. (2021): *"Score-Based Generative Modeling through Stochastic Differential Equations"*.

The forward process is described by an SDE:
$$dz = -\frac{1}{2}\beta(t)\, z\, dt + \sqrt{\beta(t)}\, dW_t$$

For the **variance-exploding** (VE) SDE, the solution is simply $z(t) = x_0 + \sigma(t)\,\varepsilon$, which is the continuous limit of our perturb-data formulation.

In [ ]:
def continuous_forward_process(x0, t, sigma_fn):
    """Apply continuous noise at time t: z(t) = x0 + sigma(t) * eps.

    Args:
        x0       : np.ndarray (D,)
        t        : float, in [0, 1]
        sigma_fn : callable, sigma_fn(t) -> float (noise level at time t)

    Returns:
        zt  : np.ndarray (D,)
        eps : np.ndarray (D,)
    """
    # TODO:
    # eps = np.random.randn(*x0.shape)
    # zt = x0 + sigma_fn(t) * eps
    # return zt, eps
    eps = np.random.randn(*x0.shape)
    zt = x0 + sigma_fn(t) * eps
    return zt, eps

In [ ]:
# Sanity check: at t=0, sigma=0 so z(0) = x0; at t=1, sigma=sigma_max so z(1) has large variance
sigma_max_cont = 5.0
sigma_fn_linear = lambda t: sigma_max_cont * t

x0_test = np.array([1.0])
for t in [0.0, 0.5, 1.0]:
    zt, eps = continuous_forward_process(x0_test, t, sigma_fn_linear)
    print(f't={t:.1f}  sigma(t)={sigma_fn_linear(t):.2f}  z(t)={zt[0]:.4f}')

In [ ]:
# TODO (bonus — exploration):
# Define sigma_fn = lambda t: sigma_max * t  (linear schedule)
# For a 1D standard normal:
# 1. Show the distribution of z(t) for t in {0, 0.25, 0.5, 0.75, 1.0}
# 2. Plot: how does the variance grow with t?
# 3. Compare: discrete schedule (L=5, 10, 50) vs continuous (n_steps=1000)
#    How does the discrete approximation converge to the continuous case?
raise NotImplementedError

---
## Summary

In this notebook you:
- Built a **geometric noise schedule** from $\sigma_{\max}$ to $\sigma_{\min}$ and perturbed data at all levels simultaneously
- Implemented a **noise-conditional score network** $s_\theta(x, \sigma)$ that conditions on $\log(\sigma)$
- Derived and implemented the **weighted DSM objective** with $\lambda(\sigma) = \sigma^2$ balancing across noise levels
- Trained the score network **jointly across all noise levels** and observed faster convergence at high $\sigma$
- Implemented **Annealed Langevin Dynamics** for progressive sample refinement from noise to data
- Proved numerically that **score-based and diffusion model objectives are equivalent** up to a sign and scale of $\sigma$
- (Bonus) Connected to the **continuous-time SDE perspective** of Song et al. (2021)

**Congratulations — you have now implemented the core families of modern generative models!**

This series covered: Gaussian distributions → GMMs → VAE → PPCA → EBMs → Diffusion → Score-based → Normalising Flows → Continuous Flows → Unified score/diffusion framework.